<a href="https://colab.research.google.com/github/italobotelho/PI2026.1/blob/main/03_ETL_Clima_Mosqlimate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 **Projeto SIEST: Sistema de Inteligência Epidemiológica e Socioespacial**
## Notebook 03: Extração de Séries Temporais Climáticas (Camada Bronze)

Este notebook é responsável por extrair o histórico climático de Campinas-SP, essencial para compreender a sazonalidade e modelar o impacto do clima (ODS 13) na eclosão de surtos epidemiológicos (Dengue, Leptospirose, etc).

**Fonte de Dados:** API REST Oficial do projeto Mosqlimate.
**Variáveis de Interesse:** Temperaturas (Mínima, Média, Máxima), Precipitação Total e Umidade Relativa do ar.

In [ ]:
!pip install mosqlient -q

### 0. Configuração da Autenticação
A API do Mosqlimate requer uma chave de utilizador (`X-UID-Key`).
* Crie uma conta em: https://mosqlimate.org/
* Aceda ao seu Perfil para gerar e copiar a sua API Key.

In [ ]:
import pandas as pd
import requests
import os
import time
from tqdm.auto import tqdm

# ⚠️ COLOQUE A SUA CHAVE AQUI DENTRO DAS ASPAS
API_KEY = 'SUA_CHAVE_API'

IBGE_CAMPINAS = 3509502
DATA_INICIO = '2014-01-01'
DATA_FIM = '2026-12-31'

# Definição do destino na Camada Bronze
PASTA_BRONZE = 'dados_clima_parquet'
os.makedirs(PASTA_BRONZE, exist_ok=True)

# Headers exigidos pelo contrato da API do Mosqlimate
HEADERS = {
    'accept': 'application/json',
    'X-UID-Key': API_KEY
}

### 1. Motor de Extração e Paginação Automática

A infraestrutura do Mosqlimate limita a devolução a 100 registos por requisição para evitar sobrecarga (*Rate Limiting*). O motor de extração abaixo implementa um sistema de paginação dinâmica (varrendo página a página) com pausas programadas (`time.sleep`) para garantir que descarregamos os 13 anos de dados diários de forma ética e sem sofrer bloqueios (*timeouts*).

In [ ]:
print("A iniciar extração climática da API Mosqlimate...")
url_base = 'https://api.mosqlimate.org/api/datastore/climate/'

todos_dados = []
pagina_atual = 1
itens_por_pagina = 100

# Inicializa a barra de progresso (sem limite conhecido inicialmente)
pbar = tqdm(desc="A baixar páginas da API")

while True:
    params = {
        'geocode': IBGE_CAMPINAS,
        'start': DATA_INICIO,
        'end': DATA_FIM,
        'page': pagina_atual,
        'per_page': itens_por_pagina
    }

    try:
        resposta = requests.get(url_base, headers=HEADERS, params=params, timeout=20)
    except Exception as e:
        print(f"\n❌ Erro de conexão: {e}")
        break

    if resposta.status_code != 200:
        print(f"\n❌ Erro (Código {resposta.status_code}): {resposta.text}")
        break

    dados_json = resposta.json()
    lista_itens = dados_json.get('items', [])

    if not lista_itens:
        break

    todos_dados.extend(lista_itens)

    # Atualiza a barra de progresso visual
    pbar.update(1)

    paginacao = dados_json.get('pagination', {})
    total_pages = paginacao.get('total_pages')
    if total_pages and pagina_atual >= total_pages:
        break

    pagina_atual += 1
    time.sleep(0.2)

pbar.close() # Fecha a barra de progresso
print(f"✅ Extração concluída! Total de {len(todos_dados)} medições diárias recolhidas.")

### 2. Transformação e Persistência (Camada Bronze)

Após a consolidação do JSON em memória, o *DataFrame* é limpo. As colunas são renomeadas para o padrão do nosso *Data Warehouse* e os dados são comprimidos no formato colunar `.parquet` para preservar espaço em disco e garantir alta velocidade na leitura futura.

In [ ]:
if todos_dados:
    df_clima = pd.DataFrame(todos_dados)

    # Agora o Pandas vai encontrar o 'geocode' sem dar KeyError
    colunas_alvo = ['date', 'temp_min', 'temp_med', 'temp_max', 'precip_tot', 'umid_med']
    df_clima = df_clima[colunas_alvo].copy()

    # Padronização de chaves para o cruzamento futuro
    df_clima.rename(columns={
        'date': 'DT_MEDICAO',
        'temp_min': 'TEMP_MIN',
        'temp_med': 'TEMP_MED',
        'temp_max': 'TEMP_MAX',
        'precip_tot': 'PRECIP_TOTAL',
        'umid_med': 'UMIDADE_MED'
    }, inplace=True)

    # Transformar Texto em Formato de Data Oficial
    df_clima['DT_MEDICAO'] = pd.to_datetime(df_clima['DT_MEDICAO'])

    # Ordenar cronologicamente por segurança
    df_clima.sort_values('DT_MEDICAO', inplace=True)

    caminho_ficheiro = f"{PASTA_BRONZE}/clima_campinas_{DATA_INICIO[:4]}_{DATA_FIM[:4]}.parquet"
    df_clima.to_parquet(caminho_ficheiro, index=False, compression='snappy')

    print(f"\n✅ PIPELINE CLIMÁTICO FECHADO! Base guardada em: {caminho_ficheiro}")
    print("\nPrévia de 5 dias climáticos (Com o IBGE validado):")
    print(df_clima.head())
else:
    print("\n⚠️ Nenhum dado foi processado. Extração falhou.")